In [7]:
import pandas as pd
import requests
import json

In [8]:
#Lista de Códigos de INEGI
Series_INEGI = ["444603", #Desocupacion
          "444610", #Ocupacion_S_Informal
          "668936", #Subocupacion
          "444602", #Participacion
          "444611", #Participacion_32C
          "444612", #Desocupacion_32C
          "444619"] #Informalidad_32C

In [10]:
#Lista de nombres de las series a importar
Nombres =  ['Desocupacion', 
            'Ocupacion_S_Informal',
            'Subocupacion',
            'Participacion', 
            'Participacion_32C',
            'Desocupacion_32C', 
            'Informalidad_32C']

In [11]:
# Series INEGI
Empleo_data = pd.DataFrame({
    'Series' : Series_INEGI,
    'Nombres': Nombres
})

In [ ]:
#Función para importar series de INEGI
token_INEGI = 'fe7e4076-9cb3-'

def importador_INEGI(serie: str, nombre: str, token_INEGI: str): 
    
    from pandas import Series

    url_api = f"https://www.inegi.org.mx/app/api/indicadores/desarrolladores/jsonxml/INDICATOR/{serie}/es/00/false/BIE-BISE/2.0/{token_INEGI}?type=json"

    #Llamado
    respuesta = requests.get(url_api)
    estatus = respuesta.status_code

    if estatus == 200:
        
        contenido = json.loads(respuesta.content)
        Series = contenido['Series'][0]['OBSERVATIONS']
        
        #Obtención de la lista de observaciones 
        Observaciones=[]
        fechas = []
        for obs in Series:  Observaciones.append(float(obs['OBS_VALUE']))
        for fecha in Series:  fechas.append(str(fecha['TIME_PERIOD']))
    
    db = pd.DataFrame({
        nombre : Observaciones,
        'fecha' : fechas
    })
    
    db['fecha'] = db['fecha'].str[:4] + "-" + db['fecha'].str[-2:] + "-01"
    db['fecha'] = pd.to_datetime(db['fecha'])
    db = db.sort_values(by = 'fecha',ascending=True).reset_index(drop = True)
    
    return db
    

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

resultados = {}

with ThreadPoolExecutor(max_workers= 10) as executor:
    
    futures = {
        executor.submit(importador_INEGI, serie, nombre, token_INEGI): (serie, nombre)
        for serie, nombre in zip(Empleo_data['Series'],Empleo_data['Nombres'])
        
    }
    
    for future in as_completed(futures):
        
        serie, nombre = futures[future]
        
        resultados[nombre] = future.result()
        